# Label the frame pairs, and split into 5 CV folds + a locked test set

Reads `feature_dataset_all.csv` from Drive, picks a threshold for each teacher score,
labels every row, then writes **6 files**: five train/val fold files and one test file.

## Files written

```
GazeVLM/folds/
├── train_val_fold1.csv     each holds ALL non-test rows,
├── train_val_fold2.csv     with split = train / val
├── train_val_fold3.csv     differing per fold
├── train_val_fold4.csv
├── train_val_fold5.csv
└── test.csv                held out once, never in any fold
```

Plus `feature_dataset_all_labeled.csv` — the master, with `fold` and `is_test` columns, so
the folds can be regenerated without re-running the analysis.

## Columns added

| Column | Values |
|---|---|
| `quad` | `0..3` — bitmask, `2*frame_high + gaze_high` |
| `quad_label` | `TRANSITION` / `PURSUIT` / `REFIXATION` / `STABLE` |
| `gate` | `SEND` / `DISCARD` |
| `frame_high`, `gaze_high` | the two bits, separately |
| `fold` | `0..4`, or `-1` for test rows |
| `is_test` | `0` / `1` |
| `split` | `train` / `val` — **only in the fold files**, different in each |

## The 2 × 2

| | gaze HIGH | gaze LOW |
|---|---|---|
| **frame HIGH** | `STABLE` (3) — **DISCARD** | `REFIXATION` (2) — SEND |
| **frame LOW** | `PURSUIT` (1) — SEND | `TRANSITION` (0) — SEND |

## Two rules the split obeys

**Everything is grouped by video.** Consecutive rows are one second apart and
near-duplicates, so a row-level split — or a row-level fold — would put almost identical
footage on both sides and inflate every metric.

**Test is removed before the folds are drawn**, and never appears in any of them. Fold
validation is for choosing thresholds, architecture and hyperparameters; test is the one
number you report, touched once.

## 1 — Mount Drive and load the two columns that matter

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"
CSV_IN    = os.path.join(DRIVE_DIR, "feature_dataset_all.csv")
CSV_OUT   = os.path.join(DRIVE_DIR, "feature_dataset_all_labeled.csv")
FOLD_DIR  = os.path.join(DRIVE_DIR, "folds")
os.makedirs(FOLD_DIR, exist_ok=True)

assert os.path.exists(CSV_IN), f"not found: {CSV_IN}"
print(f"input : {CSV_IN}  ({os.path.getsize(CSV_IN)/1e6:.1f} MB)")
print(f"folds : {FOLD_DIR}/")

# Only what thresholding needs. The packed string columns are >95% of the bytes.
NEED = ["sequence", "frame_similarity", "gaze_patch_token_sim"]

t0 = time.time()
lite = pd.read_csv(CSV_IN, usecols=NEED)
print(f"\nread {len(lite):,} rows x {len(NEED)} columns in {time.time()-t0:.1f}s")
print(f"   videos          : {lite['sequence'].nunique()}")
print(f"   in memory       : {lite.memory_usage(deep=True).sum()/1e6:.1f} MB"
      f"   ({100*lite.memory_usage(deep=True).sum()/os.path.getsize(CSV_IN):.0f}% of the file)")

assert lite[NEED[1:]].notna().all().all(), "NaN in the score columns"
for c in NEED[1:]:
    print(f"   {c:22s} mean {lite[c].mean():.3f}  std {lite[c].std():.3f}  "
          f"range [{lite[c].min():.3f}, {lite[c].max():.3f}]")
print("\n   The two spreads differ, which is why they need SEPARATE thresholds.")

## 2 — Hold out test, then draw 5 folds

Order matters: **test is removed first**, so no fold can ever contain a test video.

Videos are sorted by mean `frame_similarity` — a proxy for how static or busy the scene is
— and then dealt round-robin. Sorting first and dealing every *n*-th means every fold spans
the full range of scene types. A plain contiguous slice would hand one fold all the quiet
videos and another all the chaotic ones.

Each of the 5 fold files contains **all non-test rows**; only the `split` column changes,
marking that fold's videos as `val` and the other four folds' as `train`.

In [ ]:
K          = 5      # number of CV folds
TEST_EVERY = 6      # every 6th video (in activity order) is held out for test -> ~17%

order = lite.groupby("sequence")["frame_similarity"].mean().sort_values().index.tolist()

test_videos = order[::TEST_EVERY]                       # spread across the activity range
pool        = [v for v in order if v not in set(test_videos)]
fold_of     = {v: i % K for i, v in enumerate(pool)}     # pool is still sorted -> stratified

lite["is_test"] = lite["sequence"].isin(test_videos).astype(int)
lite["fold"]    = lite["sequence"].map(fold_of).fillna(-1).astype(int)

print(f"{len(order)} videos -> {len(test_videos)} test + {len(pool)} pooled into {K} folds\n")
print(f"   TEST    {len(test_videos):3d} videos   {int(lite.is_test.sum()):6,} rows   "
      f"({100*lite.is_test.mean():4.1f}%)")
for k in range(K):
    m = lite.fold == k
    print(f"   fold {k}  {sum(v==k for v in fold_of.values()):3d} videos   "
          f"{int(m.sum()):6,} rows   ({100*m.mean():4.1f}%)")

print("\nmean scores per group (close values = the stratification worked):")
grp = np.where(lite.is_test == 1, "test", "fold " + lite.fold.astype(str))
print(pd.DataFrame({"frame": lite.frame_similarity, "gaze": lite.gaze_patch_token_sim,
                    "g": grp}).groupby("g").mean().round(4))

# thresholds come from the POOL (everything that is not test) -- see section 4
train_pool = lite[lite.is_test == 0]
print(f"\nthresholds will use the {len(train_pool):,} pooled rows; test is untouched")

## 3 — Diagnostics: is the median the right cut?

Three questions, in order of how much they change the answer.

1. **Is either distribution bimodal?** If yes, a data-driven cut (Otsu / GMM) is
   principled. If unimodal — the usual case for cosine similarity — the median is honest
   and anything fancier is decoration around the same number.
2. **How much of the score is scene identity?** The intraclass correlation is
   between-video variance ÷ total variance. Above ~0.4 a global threshold largely encodes
   *which video this is*, and since every split is by video, the folds would then differ
   in class balance for reasons that have nothing to do with motion.
3. **Does the joint distribution leave `REFIXATION` usable?** Median cuts guarantee 50/50
   on each axis and nothing about the four cells.

In [ ]:
def icc1(x, g):
    # One-way ICC: between-video variance as a share of the total, corrected for
    # unequal group sizes. 0 = video identity says nothing, 1 = it says everything.
    d = pd.DataFrame({"x": np.asarray(x, float), "g": np.asarray(g)})
    k, N = d.g.nunique(), len(d)
    if k < 2:
        return float("nan")
    gm = d.groupby("g")["x"].agg(["mean", "count"])
    msb = (gm["count"] * (gm["mean"] - d.x.mean()) ** 2).sum() / (k - 1)
    msw = sum(((s.x - s.x.mean()) ** 2).sum() for _, s in d.groupby("g")) / (N - k)
    n0 = (N - (gm["count"] ** 2).sum() / N) / (k - 1)
    return float((msb - msw) / (msb + (n0 - 1) * msw))


def bimodal_bic(x):
    # 2-component GMM vs 1. Negative delta => two components fit better, i.e. there
    # really are two humps and a data-driven cut means something.
    from sklearn.mixture import GaussianMixture
    v = np.asarray(x, float).reshape(-1, 1)
    return (GaussianMixture(2, random_state=0).fit(v).bic(v)
            - GaussianMixture(1, random_state=0).fit(v).bic(v))


print("=" * 76)
for c in ("frame_similarity", "gaze_patch_token_sim"):
    icc, dbic = icc1(train_pool[c], train_pool["sequence"]), bimodal_bic(train_pool[c])
    print(f"{c}")
    print(f"   ICC (scene identity share) : {icc:+.3f}   "
          f"{'<- HIGH: consider PER_VIDEO = True' if icc > 0.4 else 'ok'}")
    print(f"   BIC(2) - BIC(1)            : {dbic:+,.0f}   "
          f"{'bimodal -- a data-driven cut is meaningful' if dbic < 0 else 'unimodal -- median is honest'}")
    print(f"   median                     : {train_pool[c].median():.4f}")
print("=" * 76)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
for a, c, col in zip(ax[:2], ("frame_similarity", "gaze_patch_token_sim"),
                     ("tab:blue", "tab:orange")):
    a.hist(train_pool[c], bins=60, color=col, alpha=.8)
    a.axvline(train_pool[c].median(), c="k", ls="--", lw=1.2,
              label=f"median {train_pool[c].median():.3f}")
    a.set_xlabel(c); a.set_ylabel("rows"); a.legend()
    a.set_title(f"{c}\n(one hump = median; two = consider GMM)")

ax[2].scatter(lite.frame_similarity, lite.gaze_patch_token_sim, s=2, alpha=.08)
ax[2].axvline(train_pool.frame_similarity.median(), c="k", lw=.9, ls="--")
ax[2].axhline(train_pool.gaze_patch_token_sim.median(), c="k", lw=.9, ls="--")
ax[2].set_xlabel("frame_similarity"); ax[2].set_ylabel("gaze_patch_token_sim")
r = float(np.corrcoef(lite.frame_similarity, lite.gaze_patch_token_sim)[0, 1])
ax[2].set_title(f"joint distribution — r = {r:+.3f}\n(positive r starves the off-diagonal cells)")
plt.tight_layout(); plt.show()

print(f"correlation between the two scores: r = {r:+.3f}")
print("   The more positive this is, the emptier REFIXATION and PURSUIT will be.")

### Sensitivity: how the four cells respond to the cut

Never commit to one threshold pair without seeing its neighbours. If the class shares swing
wildly across nearby percentiles, the labelling is fragile and any downstream result is
really a statement about the cut.

In [ ]:
NAMES = {0: "TRANSITION", 1: "PURSUIT", 2: "REFIXATION", 3: "STABLE"}

def shares(tf, tg, d=None):
    d = lite if d is None else d
    q = 2 * (d.frame_similarity.to_numpy() > tf).astype(int) \
        + (d.gaze_patch_token_sim.to_numpy() > tg).astype(int)
    return {NAMES[i]: 100 * float((q == i).mean()) for i in range(4)}

rows = []
for fp in (40, 50, 60):
    for gp in (40, 50, 60):
        tf = np.percentile(train_pool.frame_similarity, fp)
        tg = np.percentile(train_pool.gaze_patch_token_sim, gp)
        s = shares(tf, tg)
        rows.append(dict(frame_pct=fp, gaze_pct=gp, tau_frame=round(tf, 4),
                         tau_gaze=round(tg, 4), **{k: round(v, 1) for k, v in s.items()},
                         min_cell=round(min(s.values()), 1)))
sens = pd.DataFrame(rows)
display(sens)

best = sens.loc[sens.min_cell.idxmax()]
print(f"most balanced pair on this grid: frame_pct {best.frame_pct:.0f} / "
      f"gaze_pct {best.gaze_pct:.0f}  ->  smallest cell {best.min_cell:.1f}%")
print("   (balance is one goal among several -- see the note under section 4)")

## 4 — Choose the thresholds

**Computed once, on the pooled non-test rows, and used for every fold.**

The stricter alternative is to recompute per fold from that fold's training videos. It is
not done here, for two reasons: the labels would then differ between folds, so fold scores
would no longer be comparable; and the quantity leaked is tiny — two percentiles of the
label distribution, not anything fitted to the input features. The cell below measures
that leak by printing what each fold's own threshold *would* have been. If those numbers
scatter widely, revisit the decision.

**Test is excluded throughout**, which is the leak that would actually matter.

Two reasons to move off the medians, both visible in section 3:

- **`REFIXATION` under ~5%** — the class that justifies having two thresholds at all. To
  grow it: **lower** `FRAME_PCT`, **raise** `GAZE_PCT`.
- **High ICC** — set `PER_VIDEO = True` to take percentiles within each video. That removes
  the scene confound but changes what the label *means*, from "dissimilar" to "more
  dissimilar than usual **for this scene**" — not what a fixed-threshold gate does at
  deployment. Use it knowingly.

In [ ]:
FRAME_PCT = 50        # percentile for tau_frame
GAZE_PCT  = 50        # percentile for tau_gaze
PER_VIDEO = False     # True -> percentiles within each video (see note above)
DEAD_ZONE = 0.0       # e.g. 5.0 marks rows within +/-5 percentile of a cut as AMBIGUOUS
MIN_CELL  = 5.0       # warn if any quadrant falls below this share (%)

if not PER_VIDEO:
    TAU_F = float(np.percentile(train_pool.frame_similarity, FRAME_PCT))
    TAU_G = float(np.percentile(train_pool.gaze_patch_token_sim, GAZE_PCT))
    print(f"GLOBAL thresholds from {len(train_pool):,} pooled rows")
    print(f"   tau_frame = {TAU_F:.4f}   (p{FRAME_PCT})")
    print(f"   tau_gaze  = {TAU_G:.4f}   (p{GAZE_PCT})")
    tf_row = np.full(len(lite), TAU_F)
    tg_row = np.full(len(lite), TAU_G)

    # how much does using the pool instead of each fold's own train set actually change?
    print("\n   per-fold thresholds, had each been computed on its own training videos:")
    spread = []
    for k in range(K):
        tr = train_pool[train_pool.fold != k]
        a = np.percentile(tr.frame_similarity, FRAME_PCT)
        b = np.percentile(tr.gaze_patch_token_sim, GAZE_PCT)
        spread.append((a, b))
        print(f"      fold {k}: tau_frame {a:.4f}  ({a-TAU_F:+.4f})   "
              f"tau_gaze {b:.4f}  ({b-TAU_G:+.4f})")
    sp = np.array(spread)
    print(f"   max drift: frame {np.abs(sp[:,0]-TAU_F).max():.4f}, "
          f"gaze {np.abs(sp[:,1]-TAU_G).max():.4f}"
          f"   -> {'negligible' if max(np.abs(sp[:,0]-TAU_F).max(), np.abs(sp[:,1]-TAU_G).max()) < 0.01 else 'LARGE: recompute per fold'}")
else:
    tf_map = train_pool.groupby("sequence").frame_similarity.quantile(FRAME_PCT / 100)
    tg_map = train_pool.groupby("sequence").gaze_patch_token_sim.quantile(GAZE_PCT / 100)
    # test videos are absent from the pool, so fall back to the pooled percentile there
    TAU_F = float(np.percentile(train_pool.frame_similarity, FRAME_PCT))
    TAU_G = float(np.percentile(train_pool.gaze_patch_token_sim, GAZE_PCT))
    tf_row = lite.sequence.map(tf_map).fillna(TAU_F).to_numpy()
    tg_row = lite.sequence.map(tg_map).fillna(TAU_G).to_numpy()
    print(f"PER-VIDEO thresholds: {len(tf_map)} pairs; test videos use the pooled value")
    print("   NOTE: the label now means 'relative to this scene', not an absolute level.")

quad = 2 * (lite.frame_similarity.to_numpy() > tf_row).astype(int) \
       + (lite.gaze_patch_token_sim.to_numpy() > tg_row).astype(int)
lite["quad"] = quad

counts = pd.Series(quad).value_counts().reindex(range(4), fill_value=0)
print("\n" + "=" * 62)
print(f"{'code':>4}  {'label':<12} {'rows':>8} {'share':>8}   gate")
print("-" * 62)
for i in range(4):
    print(f"{i:>4}  {NAMES[i]:<12} {counts[i]:>8,} {100*counts[i]/len(lite):>7.1f}%   "
          f"{'DISCARD' if i == 3 else 'SEND'}")
print("=" * 62)

low = [NAMES[i] for i in range(4) if 100 * counts[i] / len(lite) < MIN_CELL]
if low:
    print(f"\n!! below {MIN_CELL}%: {low}")
    if "REFIXATION" in low:
        print("   REFIXATION is the case that justifies two thresholds instead of one.")
        print(f"   To grow it: LOWER FRAME_PCT (now {FRAME_PCT}) and RAISE GAZE_PCT "
              f"(now {GAZE_PCT}), then re-run this cell.")
    if "PURSUIT" in low:
        print("   PURSUIT is suppressed by the coarse 7x7 grid -- one patch covers ~25 deg")
        print("   of FOV, so the 'attended region' carries a lot of background. A small")
        print("   count is expected and is NOT evidence that pursuit does not happen.")
else:
    print(f"\nall four quadrants above {MIN_CELL}%")

print("\n2 x 2 (rows = frame, cols = gaze):")
display(pd.DataFrame([[counts[3], counts[2]], [counts[1], counts[0]]],
                     index=["frame HIGH", "frame LOW"], columns=["gaze HIGH", "gaze LOW"]))

### Class balance across the folds and the test set

If one fold's validation class mix differs sharply from the others, its score is not
comparable and the CV mean would be misleading.

In [ ]:
grp = np.where(lite.is_test == 1, "test", "fold " + lite.fold.astype(str))
bal = (pd.crosstab(grp, lite["quad"], normalize="index") * 100) \
        .reindex(columns=range(4), fill_value=0).rename(columns=NAMES).round(1)
display(bal)

fold_rows = bal.loc[[i for i in bal.index if i.startswith("fold")]]
sw = (fold_rows.max() - fold_rows.min())
print("spread across folds (max - min, percentage points):")
print(sw.round(1).to_string())
print(f"\n   worst spread: {sw.max():.1f} pp on {sw.idxmax()}   "
      f"{'ok' if sw.max() < 10 else '<- folds are not comparable; consider stratifying folds on quad'}")

pv = (lite.groupby("sequence")["quad"].value_counts(normalize=True)
      .unstack(fill_value=0) * 100).reindex(columns=range(4), fill_value=0).rename(columns=NAMES)
pv["max_share"] = pv.max(axis=1).round(1)
skew = pv[pv.max_share > 80]
print(f"\nvideos where one class holds >80% of rows: {len(skew)} of {len(pv)}")
if len(skew):
    print("   Those scenes the threshold cannot discriminate within. If most videos look")
    print("   like this, the label is tracking scene identity -- set PER_VIDEO = True.")

bal.plot(kind="bar", figsize=(11, 4), ylabel="% of rows",
         title="class balance per fold and test").tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 5 — Write the master, the 5 fold files and the test file

Only now is the full CSV read. Label columns come from `lite`, which is row-aligned with
it, so nothing is recomputed and nothing can drift.

Each fold file holds **all non-test rows**, with `split` marking that fold's videos `val`
and the rest `train`. Five near-copies of the data is a little wasteful on disk, but it
makes each fold a single self-contained file to hand to a trainer.

In [ ]:
t0 = time.time()
full = pd.read_csv(CSV_IN)
print(f"read {len(full):,} rows x {full.shape[1]} columns in {time.time()-t0:.1f}s "
      f"({full.memory_usage(deep=True).sum()/1e6:.0f} MB in memory)")

assert len(full) == len(lite), "row count changed between the two reads"
assert (full["sequence"].to_numpy() == lite["sequence"].to_numpy()).all(), "row order differs"
assert np.allclose(full["frame_similarity"], lite["frame_similarity"]), "score mismatch"
print("   [PASS] the two reads are row-aligned")

q = lite["quad"].to_numpy()
full["frame_high"] = (q >> 1).astype(int)
full["gaze_high"]  = (q & 1).astype(int)
full["quad"]       = q
full["quad_label"] = full["quad"].map(NAMES)
full["gate"]       = np.where(full["quad"] == 3, "DISCARD", "SEND")
full["fold"]       = lite["fold"].to_numpy()
full["is_test"]    = lite["is_test"].to_numpy()

if DEAD_ZONE > 0:
    lo_f, hi_f = (np.percentile(train_pool.frame_similarity, FRAME_PCT - DEAD_ZONE),
                  np.percentile(train_pool.frame_similarity, FRAME_PCT + DEAD_ZONE))
    lo_g, hi_g = (np.percentile(train_pool.gaze_patch_token_sim, GAZE_PCT - DEAD_ZONE),
                  np.percentile(train_pool.gaze_patch_token_sim, GAZE_PCT + DEAD_ZONE))
    amb = (full.frame_similarity.between(lo_f, hi_f) |
           full.gaze_patch_token_sim.between(lo_g, hi_g))
    full.loc[amb, ["quad_label", "gate"]] = "AMBIGUOUS"
    print(f"   DEAD_ZONE {DEAD_ZONE}%: {int(amb.sum()):,} rows AMBIGUOUS ({100*amb.mean():.1f}%)")

# ---- master ------------------------------------------------------------------
full.to_csv(CSV_OUT, index=False)
print(f"\nmaster  -> {CSV_OUT}   ({os.path.getsize(CSV_OUT)/1e6:.1f} MB)")

# ---- test --------------------------------------------------------------------
test_df = full[full.is_test == 1].copy()
test_df["idx"] = range(len(test_df))
p = os.path.join(FOLD_DIR, "test.csv")
test_df.to_csv(p, index=False)
print(f"test    -> {p}   {len(test_df):,} rows / "
      f"{test_df['sequence'].nunique()} videos   ({os.path.getsize(p)/1e6:.1f} MB)")

# ---- the five folds ----------------------------------------------------------
print()
written = []
for k in range(K):
    d = full[full.is_test == 0].copy()
    d["split"] = np.where(d["fold"] == k, "val", "train")
    d["idx"] = range(len(d))
    p = os.path.join(FOLD_DIR, f"train_val_fold{k+1}.csv")
    d.to_csv(p, index=False)
    written.append(p)
    nt, nv = int((d.split == "train").sum()), int((d.split == "val").sum())
    vt, vv = d[d.split == "train"].sequence.nunique(), d[d.split == "val"].sequence.nunique()
    print(f"fold {k+1}  -> {os.path.basename(p):<22} "
          f"train {nt:6,} rows / {vt:3d} vids   val {nv:5,} rows / {vv:2d} vids   "
          f"({os.path.getsize(p)/1e6:.1f} MB)")

tot = sum(os.path.getsize(x) for x in written) + os.path.getsize(os.path.join(FOLD_DIR, "test.csv"))
print(f"\n{K} folds + test = {tot/1e6:.0f} MB on Drive")

json.dump(dict(tau_frame=TAU_F, tau_gaze=TAU_G, frame_pct=FRAME_PCT, gaze_pct=GAZE_PCT,
               per_video=PER_VIDEO, dead_zone=DEAD_ZONE, k_folds=K,
               test_every=TEST_EVERY, computed_on="pooled non-test rows",
               n_pool_rows=int(len(train_pool)), names=NAMES,
               test_videos=sorted(test_videos), fold_of=fold_of,
               source=os.path.basename(CSV_IN)),
          open(os.path.join(DRIVE_DIR, "thresholds.json"), "w"), indent=2)
print(f"thresholds + fold assignment -> {DRIVE_DIR}/thresholds.json")

## 6 — Verify the files on disk

Reads every file back. The checks that matter most are the last three: a test video leaking
into a fold, or a video appearing in two folds' validation sets, would silently invalidate
the whole cross-validation.

In [ ]:
te = pd.read_csv(os.path.join(FOLD_DIR, "test.csv"),
                 usecols=["sequence", "quad", "quad_label", "gate", "is_test"])
fds = [pd.read_csv(os.path.join(FOLD_DIR, f"train_val_fold{k+1}.csv"),
                   usecols=["sequence", "quad", "quad_label", "gate", "split", "fold"])
       for k in range(K)]

test_set = set(te.sequence.unique())
val_sets = [set(d[d.split == "val"].sequence.unique()) for d in fds]
all_val  = set().union(*val_sets)
pool_set = set(fds[0].sequence.unique())

checks = [
    ("row counts add up",
     len(te) + len(fds[0]) == len(lite)),
    ("quad in 0..3 everywhere",
     all(d["quad"].between(0, 3).all() for d in fds + [te])),
    ("quad_label matches quad",
     all((d[d.quad_label != "AMBIGUOUS"]["quad"].map(NAMES)
          == d[d.quad_label != "AMBIGUOUS"].quad_label).all() for d in fds + [te])),
    ("gate == DISCARD iff quad == 3",
     all(((d["quad"] == 3) == (d.gate == "DISCARD")).all() for d in fds + [te])
     if DEAD_ZONE == 0 else True),
    ("every fold file has the same rows",
     all(len(d) == len(fds[0]) for d in fds)),
    ("NO test video appears in any fold",
     not (test_set & pool_set)),
    ("each pooled video is validated exactly once",
     all_val == pool_set and sum(len(s) for s in val_sets) == len(pool_set)),
    ("train and val never share a video within a fold",
     all(not (set(d[d.split=='train'].sequence) & set(d[d.split=='val'].sequence)) for d in fds)),
]
for n, v in checks:
    print(f"   [{'PASS' if v else 'FAIL'}]  {n}")
print("\n" + ("ALL PASS" if all(v for _, v in checks) else "!!! CHECK THE FAILURES ABOVE"))

print(f"\n{len(test_set)} test videos, held out of all {K} folds:")
for s in sorted(test_set):
    print(f"   {s}")

print("\nper-fold validation videos:")
for k, s in enumerate(val_sets):
    print(f"   fold {k+1}: {len(s):2d} videos, {int((fds[k].split=='val').sum()):5,} rows")

print("\nlabel distribution in the test set:")
display(te.quad_label.value_counts().rename("rows").to_frame()
        .assign(pct=lambda d: (100 * d.rows / len(te)).round(1)))

keep = 100 * (te.gate == "SEND").mean()
print(f"on test, the oracle gate sends {keep:.1f}% and skips {100-keep:.1f}%.")
print("   That skip rate is the ceiling on compute saved; a learned gate approaches it.")

---

## How to use these

```python
FOLDS = "/content/drive/MyDrive/GazeVLM/folds"

for k in range(1, 6):
    df = pd.read_csv(f"{FOLDS}/train_val_fold{k}.csv")
    train, val = df[df.split == "train"], df[df.split == "val"]
    ...                       # fit on train, score on val
# average the five val scores -> that is your CV estimate

test = pd.read_csv(f"{FOLDS}/test.csv")     # ONCE, at the very end
```

Each fold file is self-contained: the same rows, only `split` differs. The existing
trainers split internally by sequence, so to use these you would pass the fold file and
have them read `split` instead of calling `split_by_sequence`.

## What the CV buys you

A single train/val split of ~14 videos gives one noisy number. Five folds give five, and
their **spread** tells you how much of any result is the particular videos that landed in
validation. Given that the `r = −0.719` finding is still shadowed by a two-cluster
concern, that spread is the thing worth reading.

## Rules

| | |
|---|---|
| **Test** | Look at it once, at the end. Every glance spends its value |
| **Thresholds** | Fixed across folds, from pooled non-test rows. Section 4 prints how far each fold's own threshold would have drifted — if that is large, revisit |
| **`quad` is a target** | It comes from the true DINOv2 similarities. The gate has to predict these from gaze alone, so never feed `quad`, `gate` or either similarity as a feature |

## Regenerating

`thresholds.json` stores the cut, the test video list and the full fold assignment, so the
same folds can be rebuilt from the master CSV without re-running the analysis.